In [8]:
import json
import pandas as pd
import numpy as np

In [9]:
import pandas as pd
import json

df = pd.read_csv('../data/senators/senators_a-c_wip.csv')


def parse_congresses(val):
    if pd.isna(val) or val == '':
        return []
    try:
        return json.loads(val)
    except:
        return []

df['congresses_parsed'] = df['congresses'].apply(parse_congresses)

# cleaning df
df['num_terms']      = df['congresses_parsed'].apply(len)
df['positions']      = df['congresses_parsed'].apply(lambda x: list({c.get('position') for c in x}))
df['states']         = df['congresses_parsed'].apply(lambda x: list({c.get('stateName') for c in x}))
df['parties']        = df['congresses_parsed'].apply(lambda x: list({p for c in x for p in c.get('parties', [])}))
df['congress_range'] = df['congresses_parsed'].apply(lambda x: f"{min(c.get('congressNumber',0) for c in x)}-{max(c.get('congressNumber',0) for c in x)}" if x else '')

df = df.drop(columns=['congresses', 'congresses_parsed'])


In [11]:
df = df.drop(columns = ['unaccentedGivenName','unaccentedMiddleName','unaccentedFamilyName','nickName', 'honorificPrefix', 'honorificTitle',])

In [13]:
df.to_csv('senators/senators_a-c.csv', index=False)

In [17]:
df_clean = pd.read_csv("../data/senators/senators_a-c.csv")
df_clean.head()

,id,firstName,lastName,middleName,honorificSuffix,birthYear,deathYear,num_terms,positions,states,parties,congress_range
0,A000001,Fred,Aandahl,George,NaN,1897.0,1966.0,1,['Representative'],['ND'],['Republican'],82-82
1,A000002,Watkins,Abbitt,Moorman,NaN,1908.0,1998.0,13,['Representative'],['VA'],['Democrat'],80-92
2,A000003,Joel,Abbot,NaN,NaN,1776.0,1826.0,4,['Representative'],['GA'],"['Democratic Republican', 'Crawford Republican']",15-18
3,A000004,Amos,Abbott,NaN,NaN,1786.0,1868.0,3,['Representative'],['MA'],['Whig'],28-30
4,A000006,Joseph,Abbott,Carter,NaN,1825.0,1881.0,2,['Senator'],['NC'],['Republican'],40-41


In [18]:
df_clean['bio'] = ''

In [42]:
import pandas as pd
import re
import random
from playwright.async_api import async_playwright

df_clean = pd.read_csv("../data/senators/senators_a-c.csv")

# Resume from checkpoint if it exists
try:
    checkpoint = pd.read_csv('senators_checkpoint.csv')
    done_ids = set(checkpoint.loc[checkpoint['bio'].notna() & (checkpoint['bio'] != ''), 'id'])
    df_clean = checkpoint
    print(f"Resuming from checkpoint: {len(done_ids)} already done")
except FileNotFoundError:
    df_clean['bio'] = ''
    done_ids = set()

def extract_bio(body_text):
    # Match text starting with an all-caps last name, e.g. "AANDAHL, Fred George, a Representative..."
    match = re.search(r'([A-Z]{2,}[A-Z\s,]+,\s[A-Z][a-z]+.*?)(?=Content & Site Support)', body_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ''

async def scrape():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        for i, row in df_clean.iterrows():
            member_id = row['id']

            if member_id in done_ids:
                continue

            url = f"https://bioguide.congress.gov/search/bio/{member_id}"

            try:
                await page.goto(url, wait_until="networkidle", timeout=30000)
                await page.wait_for_timeout(2000)

                body_text = await page.inner_text("body")
                bio = extract_bio(body_text)

                if bio:
                    print(f"✓ {member_id}")
                else:
                    print(f"~ {member_id}: page loaded but bio not found")

            except Exception as e:
                bio = ''
                print(f"✗ {member_id}: {e}")

            df_clean.at[i, 'bio'] = bio

            # Checkpoint every 50 rows
            if i % 50 == 0:
                df_clean.to_csv('senators/senators_checkpoint.csv', index=False)
                print(f"  [checkpoint saved at row {i}]")

            await page.wait_for_timeout(random.randint(1000, 2500))

        await browser.close()

    df_clean.to_csv('../data/senators/senators_a-c_with_bios.csv', index=False)
    print(f"\nDone. Saved to senators/senators_a-c_with_bios.csv")

await scrape()

Resuming from checkpoint: 1895 already done
~ A000359: page loaded but bio not found
✓ A000020: ACKER, Ephraim Leister
1827 – 1903
Biography
Congresses
Bibliography
Research Co...
✓ A000028: ADAMS, Alva Blanchard
1875 – 1941
Biography
Congresses
Bibliography
Research Col...
✓ A000059: AGNEW, Spiro Theodore
1918 – 1996
Biography
Congresses
Bibliography
Research Col...
✓ A000371: AGUILAR, Peter Rey
1979 –
Biography
Congresses
Bibliography
Research Collections...
✓ A000068: AITKEN, David Demerest
1853 – 1930
Biography
Congresses
Bibliography
Research Co...
~ A000358: page loaded but bio not found
✓ A000094: ALEXANDER, Hugh Quincy
1911 – 1989
Biography
Congresses
Bibliography
Research Co...
✓ A000102: ALEXANDER, Sydenham Benoni
1840 – 1921
Biography
Congresses
Bibliography
Researc...
✓ A000103: ALEXANDER, William Vollie (Bill), Jr.
1934 –
Biography
Congresses
Bibliography
R...
~ A000109: page loaded but bio not found
✓ A000118: ALLEN, Clifford Robertson
1912 – 1978
Biography
Congresses
Bib

In [50]:
import pandas as pd
import re

df = pd.read_csv('../data/senators/senators_a-c_with_bios.csv')

def clean_bio(bio):
    if not isinstance(bio, str) or bio.strip() == '':
        return ''
    # Cut off footer
    bio = re.split(r'Content & Site Support', bio)[0]
    # The actual bio sentence always comes after the party name on its own line
    # Split on the last occurrence of a known party label on its own line
    match = re.split(r'\n(?:Republican|Democrat|Whig|Federalist|Independent|Democratic Republican|Crawford Republican|[A-Za-z ]+)\s*\n', bio)
    if match:
        return match[-1].strip()
    return bio.strip()

df['bio'] = df['bio'].apply(clean_bio)

df.to_csv('../data/senators/senators_a-c_with_bios.csv', index=False)
print("Done. Sample output:")
print(df['bio'].head(3).to_string())

Done. Sample output:
0    AANDAHL, Fred George, a Representative from No...
1    ABBITT, Watkins Moorman, a Representative from...
2    Crawford Republican\nABBOT, Joel, a Representa...
